# 01. ESS 배터리 데이터 점검 및 탐색적 데이터 분석 (EDA)

## 1. 프로젝트 개요 및 데이터 역할 정의

### 데이터 파일 구성 ()
| 파일명 | 크기 | 식별명 | 셀 수 | 역할 및 적용 범위 |
| :--- | :--- | :--- | :---: | :--- |
|  | ~2.8 GB | **Batch 1** | 46 | **Train & 내부 Hold-out 검증용** (모델 개발 메인 데이터) |
|  | ~1.9 GB | **Batch 2** | 47 | **최종 일반화 Test용** (모델 완성 후 1회 평가) |
|  | ~3.0 GB | **Batch 3** | 46 | **추가 비교 분석용** (Batch 1·2 완료 후 확장) |
|  | ~85 MB | **Extra** | - | **본 과제 제외** (충전 최적화 연구 데이터) |

> **데이터 분석 및 모델링 원칙**:
> 1. **Batch 1 중심의 탐색 및 피처 설계**: 모델 학습 및 내부 검증은 오직 Batch 1의 데이터만을 탐색하여 설계합니다.
> 2. **Batch 2는 최종 테스트셋으로 격리**: Batch 2의 라벨이나 세부 통계를 사전에 보고 모델을 튜닝하는 **데이터 누수(Data Snooping)**를 엄격히 방지합니다.
> 3. **Batch 2 사전 점검 범위**: 결측치, 이상 프로토콜 등 "테스트셋으로서 평가 가능한 유효 셀"을 확정하는 최소 규격 점검만 수행합니다.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 프로젝트 루트 등록
project_root = os.path.abspath("..//")
if project_root not in sys.path:
    sys.path.append(project_root)

from src.preprocess import load_batch_summary

# 시각화 스타일 설정
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.family"] = "AppleGothic" if sys.platform == "darwin" else "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

# 소수점 표시 옵션 설정 (4자리 고정)
pd.set_option("display.float_format", "{:.4f}".format)

DATA_DIR = os.path.join(project_root, "data")
print(f"Data Directory: {DATA_DIR}")

Data Directory: /Users/skala_yh/DS_miniproject/data


## 2. Batch 1 (모델 개발용) 데이터 로드 및 초기 지표 추출
- 원시 요약 데이터()의 모든 필드를 포함하여 초기 100사이클 지표를 전수 추출합니다.
- 포함 필드: 방전용량(), 충전용량(), 쿨롱효율(), 내부저항(), 온도(, , ), 충전시간().

In [2]:
b1_path = os.path.join(DATA_DIR, "2017-05-12_batchdata_updated_struct_errorcorrect.mat")
df_b1 = load_batch_summary(b1_path, "Batch1")
print(f"Batch 1 로드 셀 수: {len(df_b1)}개")

def extract_full_summary_features(df):
    """원시 summary 구조체의 모든 초기(Cycle 2 & 100) 지표 추출"""
    rows = []
    for _, r in df.iterrows():
        qd = r["QDischarge"]
        qc = r["QCharge"]
        ir = r["IR"]
        tmax = r["Tmax"]
        tavg = r["Tavg"]
        tmin = r["Tmin"]
        ch_time = r["chargetime"]
        
        # 1. 용량 지표 (Cycle 2 및 Cycle 100)
        q_init = qd[1] if len(qd) > 1 else np.nan
        qc_init = qc[1] if len(qc) > 1 else np.nan
        ce_init = (q_init / qc_init * 100) if (qc_init and not np.isnan(qc_init)) else np.nan
        
        q_100 = qd[99] if len(qd) > 99 else np.nan
        qc_100 = qc[99] if len(qc) > 99 else np.nan
        ce_100 = (q_100 / qc_100 * 100) if (qc_100 and not np.isnan(qc_100)) else np.nan
        delta_q = (q_100 - q_init) if (not np.isnan(q_100) and not np.isnan(q_init)) else np.nan
        
        # 2. 내부저항 지표 (IR)
        ir_init = ir[1] if len(ir) > 1 else np.nan
        ir_100 = ir[99] if len(ir) > 99 else np.nan
        delta_ir = (ir_100 - ir_init) if (not np.isnan(ir_100) and not np.isnan(ir_init)) else np.nan
        
        # 3. 온도 지표 (초기 100사이클 통계)
        t_min_100 = np.min(tmin[:100]) if len(tmin) >= 100 else np.nan
        t_avg_100 = np.mean(tavg[:100]) if len(tavg) >= 100 else np.nan
        t_max_100 = np.max(tmax[:100]) if len(tmax) >= 100 else np.nan
        delta_t = (t_max_100 - t_min_100) if (not np.isnan(t_max_100) and not np.isnan(t_min_100)) else np.nan
        
        # 4. 충전 시간 지표
        ch_init = ch_time[1] if len(ch_time) > 1 else np.nan
        ch_100 = ch_time[99] if len(ch_time) > 99 else np.nan
        ch_avg = np.mean(ch_time[1:100]) if len(ch_time) >= 100 else np.nan
        
        rows.append({
            "cell_id": r["cell_id"],
            "policy": r["policy"],
            "cycle_life": r["cycle_life"],
            "n_cycles": r["n_cycles"],
            "q_init_c2": q_init,
            "qc_init_c2": qc_init,
            "ce_init_pct": ce_init,
            "q_c100": q_100,
            "qc_c100": qc_100,
            "ce_c100_pct": ce_100,
            "delta_q_100_2": delta_q,
            "ir_init": ir_init,
            "ir_c100": ir_100,
            "delta_ir_100_2": delta_ir,
            "t_min_100": t_min_100,
            "t_avg_100": t_avg_100,
            "t_max_100": t_max_100,
            "delta_t_100": delta_t,
            "ch_time_init_min": ch_init,
            "ch_time_c100_min": ch_100,
            "ch_time_avg_min": ch_avg
        })
    return pd.DataFrame(rows)

feat_b1 = extract_full_summary_features(df_b1)
var_map = {
    'cycle_life': ('수명 (전체 사이클 수)', 'Cycle'),
    'n_cycles': ('기록된 총 사이클 수', 'Cycle'),
    'q_init_c2': ('초기 방전용량 (2사이클)', 'Ah'),
    'qc_init_c2': ('초기 충전용량 (2사이클)', 'Ah'),
    'ce_init_pct': ('초기 쿨롱효율 (Qd/Qc)', '%'),
    'q_c100': ('100사이클 방전용량', 'Ah'),
    'qc_c100': ('100사이클 충전용량', 'Ah'),
    'ce_c100_pct': ('100사이클 쿨롱효율', '%'),
    'delta_q_100_2': ('용량 변화량 (100C - 2C)', 'Ah'),
    'ir_init': ('초기 내부저항 (2사이클)', 'Ω'),
    'ir_c100': ('100사이클 내부저항', 'Ω'),
    'delta_ir_100_2': ('내부저항 변화량', 'Ω'),
    't_min_100': ('초기 100사이클 최저온도', '°C'),
    't_avg_100': ('초기 100사이클 평균온도', '°C'),
    't_max_100': ('초기 100사이클 최고온도', '°C'),
    'delta_t_100': ('온도 편차 (최고 - 최저)', '°C'),
    'ch_time_init_min': ('초기 충전시간 (2사이클)', '분'),
    'ch_time_c100_min': ('100사이클 충전시간', '분'),
    'ch_time_avg_min': ('초기 100사이클 평균충전시간', '분')
}

Batch 1 로드 셀 수: 46개


## 3. [표] Batch 1 (학습/검증셋) 종합 기초 통계량 (한글 명칭 및 단위 포함)
- 는 정수로 표기하며, 나머지 연속형 통계량은 **소수점 넷째 자리()**로 표기합니다.
- 각 변수의 한글 명칭과 측정 단위가 함께 표기되어 데이터의 물리적 의미를 직관적으로 파악할 수 있습니다.

In [3]:
def format_describe_table(df):
    """count는 정수, 나머지 통계량은 소수점 4자리, 한글 명칭과 단위 표기"""
    desc = df.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']].copy()
    desc['count'] = desc['count'].astype(int)
    desc.insert(0, '변수 한글명', desc.index.map(lambda x: var_map.get(x, (x, ''))[0]))
    desc.insert(1, '단위', desc.index.map(lambda x: var_map.get(x, ('', ''))[1]))
    return desc.style.format({
        'count': '{:d}',
        'mean': '{:.4f}',
        'std': '{:.4f}',
        'min': '{:.4f}',
        '50%': '{:.4f}',
        'max': '{:.4f}'
    })

print('=== Batch 1 (학습/검증용) 종합 기초 통계량 (N=46) ===')
display(format_describe_table(feat_b1))

=== Batch 1 (학습/검증용) 종합 기초 통계량 (N=46) ===


,변수 한글명,단위,count,mean,std,min,50%,max
cycle_life,수명 (전체 사이클 수),Cycle,46,844.7174,184.6292,534.0000,858.5000,1227.0000
n_cycles,기록된 총 사이클 수,Cycle,46,843.7174,184.6292,533.0000,857.5000,1226.0000
q_init_c2,초기 방전용량 (2사이클),Ah,46,1.0781,0.0077,1.0538,1.0787,1.0946
qc_init_c2,초기 충전용량 (2사이클),Ah,46,1.0780,0.0078,1.0529,1.0789,1.0947
ce_init_pct,초기 쿨롱효율 (Qd/Qc),%,46,100.0058,0.0417,99.9363,99.9962,100.1030
q_c100,100사이클 방전용량,Ah,46,1.0807,0.0078,1.0600,1.0815,1.0958
qc_c100,100사이클 충전용량,Ah,46,1.0804,0.0078,1.0594,1.0811,1.0954
ce_c100_pct,100사이클 쿨롱효율,%,46,100.0286,0.0237,99.9875,100.0251,100.0936
delta_q_100_2,용량 변화량 (100C - 2C),Ah,46,0.0026,0.0030,-0.0114,0.0028,0.0066
ir_init,초기 내부저항 (2사이클),Ω,46,0.0167,0.0006,0.0153,0.0167,0.0199


## 4. 기초 통계량 특이치 분석 및 물리적 해석 (Q&A)

위 통계표에서 관측되는 3가지 특이 사항에 대한 원인 분석 및 데이터 조치 방안입니다:

### Q1. 쿨롱 효율()이 100%를 초과하는 값(최대 100.0936%)이 나오는 이유 및 정상 여부
- **정의**: 쿨롱 효율($) = $rac{Q_{	ext{discharge}}}{Q_{	ext{charge}}} 	imes 100$ (%)
- **물리적 원인**: 이상적인 화학 반응에서는 100%를 초과할 수 없으나, 실제 배터리 충·방전 시험기에서는 다음과 같은 이유로 .01\% \sim 100.09\%$ 수준의 초과가 정상적으로 발생합니다:
  1. **전(Previous) 사이클 잔류 전하 방출**: 급속 충전 후 일부 리튬 이온이 음극 구조 내에 머물다가 다음 사이클 방전 시 함께 방출되는 현상 (잔류 전하 효과).
  2. **온도 변동에 따른 적분 오차**: 충전 시점과 방전 시점의 셀 미세 표면 온도 차이로 인해 전해질 이온 전도도가 변화하여 전류 적분치($\int I dt$)가 zsh.0003 \sim 0.001\,	ext{Ah}$ 수준으로 미세하게 변동.
  3. **전류계 측정 분해능 허용 오차**: 시험 장비 센서의 정밀도 허용 한계($\pm 0.05\%$) 범위 내의 측정 특성.
- **결론**: **정상적인 물리적/계측적 현상**이며, 배터리 손상이나 결함이 아닙니다. 피처 생성 시 그대로 활용 가능합니다.

### Q2. 최저 온도()가 0.0000°C로 나오는 이유
- **물리적 배경**: 실험은 ^\circ	ext{C}$ 항온 챔버(Environmental Chamber) 내부에서 진행되었으므로 배터리 온도가 실제로 zsh^\circ	ext{C} 될 수 없습니다.
- **원인**: 원시 MATLAB 파일()을 확인한 결과, 실험 장비의 데이터 로깅 소프트웨어가 최저 온도() 채널을 별도로 기록하지 않고 기본 초기화 값인 으로 채워둔 채 저장되었습니다.
- **결론**: 실제 측정값이 아닌 **시스템 로깅 미수집 기본값(Placeholder)**입니다. 따라서 은 신뢰할 수 없는 변수이므로 모델 입력 피처에서 제외합니다.

### Q3. 최고 온도()와 온도 편차()가 36.7841°C로 완전히 일치하는 이유
- **원인**: 수식상 $	ext{delta\_t} = 	ext{t\_max\_100} - 	ext{t\_min\_100} 계산되는데, Q2에서 밝힌 바와 같이 이 zsh.0000 고정되어 있어 $	ext{t\_max\_100} - 0.0 = 	ext{t\_max\_100} 되었기 때문입니다.
- **조치**: 은 과 100% 동일한 중복 변수(다중공선성 1.0)입니다. 피처 엔지니어링 시 중복 피처로 배제하고, 실제 계측된 과 을 핵심 온도 피처로 활용합니다.

## 5. Batch 2 (최종 테스트셋) 규격 및 무결성 사전 점검
- **목적**: Batch 2의 수명 분포나 라벨 정보를 미리 보고 모델을 편향시키는 행위(Data Leakage)를 방지하면서, 오직 **"테스트 데이터셋으로서 유효한 셀"**만을 사전 정의합니다.
- 프로토콜 유형 및 결측 여부만 점검합니다.

In [4]:
b2_path = os.path.join(DATA_DIR, "2018-02-20_batchdata_updated_struct_errorcorrect.mat")
df_b2 = load_batch_summary(b2_path, "Batch2")

# Batch 2 정책별 셀 수 및 라벨 결측 현황 집계
b2_audit = df_b2.groupby("policy").agg(
    셀수=("cell_id", "count"),
    수명_결측수=("cycle_life", lambda s: s.isna().sum()),
    평균_기록사이클=("n_cycles", "mean")
).reset_index()

print(f"Batch 2 전체 로드 셀 수: {len(df_b2)}개")
print(f"  - 정상 라벨 보유 셀: {df_b2['cycle_life'].notna().sum()}개")
print(f"  - 라벨 결측(NaN) 셀: {df_b2['cycle_life'].isna().sum()}개 (평가 제외 대상)")

display(b2_audit)

Batch 2 전체 로드 셀 수: 47개
  - 정상 라벨 보유 셀: 39개
  - 라벨 결측(NaN) 셀: 8개 (평가 제외 대상)


,policy,셀수,수명_결측수,평균_기록사이클
0,3.6C(30%)-6C,2,0,453.0000
1,3.6C(80%)-3.6C(SLOWCYCLE,1,1,101.0000
2,3.6C(9%)-5C,2,0,411.5000
3,3.6C(9%)-5C(SLOWCYCLE,1,1,434.0000
4,4.65C(44%)-5C,2,0,505.0000
5,4.65C(69%)-6C,2,0,473.5000
6,4.8C(80%)-4.8C,5,0,506.6000
7,4.8C(80%)-4.8C(SLOWCYCLE,1,1,101.0000
8,4.8C(80%)-4.8C-newstructure,3,0,900.3333
9,5.2C(50%)-4.25C,5,0,472.8000
